# Demo of upcoming Stats

This notebook is to demonstrate stats not yet on the UI. The documentation for it including the development manifesto can be found [here](https://renalregistry.atlassian.net/wiki/spaces/UDF/pages/2308079624/Real-time+statistics+manifesto). 

In [1]:
from ukrdc.database import Connection
from sqlalchemy.orm import sessionmaker
import datetime as dt
import plotly.graph_objects as go

to_time = dt.datetime.now()
from_time = to_time - dt.timedelta(days=90)

engine = Connection.get_engine_from_file(key="ukrdc_staging")

ukrdc3_sessionmaker = sessionmaker(
    autocommit=False, autoflush=False, bind=engine
)

ukrdc3 = ukrdc3_sessionmaker()

# Primary Renal Diagnosis

The first major update to be due to be integrated to the UI is the demographics broken down by primary renal diagnosis.

In [2]:
from ukrdc_stats.calculators.demographics_prd import RenalDiagnosisStatsCalculator
from IPython.display import display, Markdown
import plotly.graph_objects as go 
import plotly.express as px

import pandas as pd

renal_unit = "RNJ00"
#renal_unit = "RJZ"

calculator = RenalDiagnosisStatsCalculator(ukrdc3, renal_unit)
calculator.extract_patient_cohort()


# run function to extract stats
output = calculator.extract_stats()

# run function to extract stats
output = calculator.extract_stats()
back_to_back = pd.DataFrame(output.gender.data.dict())

# Plot gender stats back to back 
data = [
    go.Bar(
        x = -back_to_back[back_to_back.x == "Male"].z,
        y = back_to_back[back_to_back.x == "Male"].y,
        orientation="h",
        name = "Male"
    ), 
    go.Bar(
        x = back_to_back[back_to_back.x == "Female"].z,
        y = back_to_back[back_to_back.x == "Female"].y,
        orientation="h",
        name = "Female"
    ) 
]

fig = go.Figure(
    data=data,
    layout = {
        "title" :{
            "text": output.gender.metadata.title,
            "x" : 0.5,
            "xanchor" : "center" 
        }
    }
)

display(Markdown(output.gender.metadata.description))

fig.show()


# Primary Renal Diagnosis by Sex

## Overview
Number of living patients registered with the renal unit, categorized by sex and primary renal diagnosis.

## Methodology
- The UKRDC's primary renal diagnosis and patient demographic information are matched to the PatientRecord.
- Patients with a recorded deathtime are excluded from the analysis.
- Optionally, patients are further excluded by matching to the date of death in NHS tracing records.
- Patients were aggregated by sex and primary renal diagnosis. Patients not matched to a renal diagnosis were assigned to the group "No PRD".
- No deduplication is carried out: a one-to-one relationship between primary renal diagnosis and age is assumed.

## UKRDC Entities Used
- [PatientRecord](https://renalregistry.atlassian.net/l/cp/KCZ6A2bX)
- [Patient](https://renalregistry.atlassian.net/l/cp/0MXHtpTU)
- [RenalDiagnosis](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2009071645/RenalDiagnosis+Diagnosis)



In [3]:
# plot age
stacked_bar_data = pd.DataFrame(output.age.data.dict())
stacked_bar_data.x = pd.to_numeric(stacked_bar_data.x)
stacked_bar_data.rename(columns={"x":"Age", "y":"Primary Renal Diagnosis", "z":"Patients"}, inplace = True)

fig = px.bar(stacked_bar_data, x="Age", y = "Patients", color = "Primary Renal Diagnosis", title = output.age.metadata.title)

fig.update_layout(title_x = 0.5)

display(Markdown(output.age.metadata.description))

fig.show()


# Primary Renal Diagnosis by Age

## Overview
Number of living patients registered with the renal unit, categorized by chronological age and primary renal diagnosis.

## Methodology
- The UKRDC's primary renal diagnosis and patient demographic data are matched to the PatientRecord.
- Patients with a recorded deathtime are excluded from the analysis.
- Optionally, patients are further excluded by matching to the date of death in NHS tracing records.
- Patient's chronological age is calculated from the date of birth on the patient record
- Patients are aggregated by age and primary renal diagnosis. Patients not matched to a renal diagnosis are assigned to the group "No PRD".
- No deduplication is carried out: a one-to-one relationship between primary renal diagnosis and age is assumed.

## UKRDC Entities Used
- [PatientRecord](https://renalregistry.atlassian.net/l/cp/KCZ6A2bX)
- [Patient](https://renalregistry.atlassian.net/l/cp/0MXHtpTU)
- [RenalDiagnosis](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2009071645/RenalDiagnosis+Diagnosis)


In [4]:
# plot ethnicity
ethnicity_data = pd.DataFrame(output.ethnic_group.data.dict())
ethnicity_data.rename(columns={"x":"Ethnic Group", "y":"Primary Renal Diagnosis", "z":"Patients"}, inplace = True)
fig = px.sunburst(
    ethnicity_data, 
    path = ["Ethnic Group", "Primary Renal Diagnosis"],
    values = "Patients",
    title = output.ethnic_group.metadata.title
)
fig.update_layout(title_x = 0.5)

display(Markdown(output.ethnic_group.metadata.description))
fig.show()


# Primary Renal Diagnosis by Ethnicity

## Overview
Number of living patients registered with the renal unit, categorized by ethnicity and primary renal diagnosis.

## Methodology
- The UKRDC's primary renal diagnosis and patient demographic information are matched to the PatientRecord.
- Patients with a recorded deathtime are excluded from the analysis.
- Optionally, patients are further excluded by matching to the date of death in NHS tracing records.
- Patients are aggregated by ethnicity and primary renal diagnosis. Patients not matched to a renal diagnosis are assigned to the group "No PRD".
- No deduplication is carried out: a one-to-one relationship between primary renal diagnosis and age is assumed.

## UKRDC Entities Used

The following UKRDC entities were used in this report:

- [PatientRecord](https://renalregistry.atlassian.net/l/cp/KCZ6A2bX)
- [Patient](https://renalregistry.atlassian.net/l/cp/0MXHtpTU)
- [RenalDiagnosis](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2009071645/RenalDiagnosis+Diagnosis)


# Time Series Treatments


In [5]:
from ukrdc_stats.calculators.dialysis import TimeSeriesTreatment

calculator = TimeSeriesTreatment(
    ukrdc3, renal_unit, time_delta=dt.timedelta(days=30), number=4, to_time=dt.datetime(2022, 12, 31)
)
output = calculator.extract_stats()

ichd = go.Bar(x = output.incident_treatment_historical.data.dates, y = output.incident_treatment_historical.data.ichd, name = "ICHD")
hhd = go.Bar(x = output.incident_treatment_historical.data.dates, y = output.incident_treatment_historical.data.hhd, name = "HHD")
xxhd = go.Bar(x = output.incident_treatment_historical.data.dates, y = output.incident_treatment_historical.data.xxhd, name = "HD Unknown")
pd = go.Bar(x = output.incident_treatment_historical.data.dates, y = output.incident_treatment_historical.data.pd, name = "PD")
tx = go.Bar(x = output.incident_treatment_historical.data.dates, y = output.incident_treatment_historical.data.tx, name = "TX")
data = [ichd, hhd, xxhd, pd, tx]
layout = go.Layout(
    barmode="stack", 
    title= {"text": output.incident_treatment_historical.metadata.title, "x":0.5})
fig = go.Figure(data=data, layout=layout)


display(Markdown(output.incident_treatment_historical.metadata.description))
fig.show()



# Treatment History
## Overview 

## Methodology

## UKRDC Entities Used 


In [6]:
ichd = go.Bar(x = output.prevalent_treatment_historical.data.dates, y = output.prevalent_treatment_historical.data.ichd, name = "ICHD")
hhd = go.Bar(x = output.prevalent_treatment_historical.data.dates, y = output.prevalent_treatment_historical.data.hhd, name = "HHD")
xxhd = go.Bar(x = output.prevalent_treatment_historical.data.dates, y = output.prevalent_treatment_historical.data.xxhd, name = "HD Unknown")
pd = go.Bar(x = output.prevalent_treatment_historical.data.dates, y = output.prevalent_treatment_historical.data.pd, name = "PD")
tx = go.Bar(x = output.prevalent_treatment_historical.data.dates, y = output.prevalent_treatment_historical.data.tx, name = "TX")

data = [ichd, hhd, xxhd, pd, tx]
layout = go.Layout(
    barmode="stack", 
    title= {"text" : output.prevalent_treatment_historical.metadata.title, "x":0.5}
)
fig2 = go.Figure(data=data, layout=layout)


display(Markdown(output.prevalent_treatment_historical.metadata.description))
fig2.show()


# Next Prevalent Patients
## Overview 

## Methodology

## UKRDC Entities Used 


In [7]:
patient_modality = go.Figure(
    data = [
        go.Sankey(
            node = dict(
                label = output.treatment_changes.node.node_labels
            ),
            link = output.treatment_changes.link.dict()
        )
    ]
)

display(Markdown(output.treatment_changes.metadata.description))
patient_modality.show()


# Next Treatment of Prevalent Patients
## Overview
This sankey plot displays the next treatment modality for aggregated patients at a specified unit, as recorded by the UKRDC.

## Methodology
- This calculator has a dependancy on the one used for patients undergoing kidney replacement therapy. The cohort is constructed in the same way, but patients are aggregated without deduplication.
- The cohort of treatment modalities is time-windowed in the same way.
- A set of events is calculated and ranked chronologically. An event can be the beginning of a treatment, a death, or a discharge.
- Patients with only one event, i.e., no discharge, new treatment, or death, are assumed to remain on the same treatment.
- The number of patients on each combination of events is aggregated.   

## UKRDC Entities Used 
The same entities the dialysis calculator dependancy these are:
- [PatientRecord](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2006450149/PatientRecord): ukrdcid, sendingextract
- [Patient](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2006450145/Patient): deathtime
- [Treatment](https://renalregistry.atlassian.net/wiki/spaces/UD/pages/2006450155/Treatment+Encounter): qbl05, hdp04, fromtime, totime, dischargereasoncode, healthcarefacilitycode
- [ModalityCodes](https://renalregistry.atlassian.net/l/cp/Ac1YeFfH): registry_code_type
